# Project 03: Gibbs Sampling (Random Algorithm)

In [2]:
import random                 # generate pseudo random numbers
import numpy as np            # works with arrays
import seqlogo                # plot sequence logos from PFMs

# import functions for reading data, building motifs, and scoring k-mers
from data_readers import *
from seq_ops import get_seq, reverse_complement
from motif_ops import *

ModuleNotFoundError: No module named 'data_readers'

---
## Implement Gibbs Sampler


Gibbs sampling is a MCMC approach to identify enrichments. Here we will implement a method to identify motifs from a set of regions. 

Important considerations:
- We will need to score each sequence with a PWM using the `score_kmer()` or `score_sequence()` functions
    - You will need to investigate into the help documentation and libraries to identify how best to use these functions.
- These sites are often not strand-specific and so both scores on the negative as well as positive strand should be considered
- To select a random sequence, use `random.randint()` or `numpy.random.randint()`
- To select a new position $m$ (as defined below) use `random.choices()` or `numpy.random.choice()`  

Assumptions: 
- We know $k$ as the length of expected motif
- Each sequence contains the motif



```
GibbsMotifFinder(DNA, k-length)
    random pick of k-length sequences from each line of DNA as Motifs
    for j ← 1 to 10000 or Motifs stops changing
        i ← Random(N) where N is number of DNA entries
        PWM ← PWM constructed from all Motifs except for Motifi
        Motifi ← select position m from PWM-scored k-mers in DNAi in probabilistic fashion from score distribution
    return PFM
```

Probability of choosing position $m = \frac{A_{m}}{\sum_{l}A_{l}}$ for positions $l$ in DNAi


**Note:** I have also added a function to `motif_ops.py` that will calculate the information content of your motifs. This is useful to observe the progression of your Gibbs sampler as well as a measure of convergence. You can use this function as `IC = pfm_ic(pfm)`. You should expect a slow increase of IC until it plateaus such as in the plot below from your lecture slides:

<center><img src='figures/Gibbs_Sampling.png'/ width=600px></center>

---
# Project Start

## Our approach

All of the steps from the lecture slides are written inside `GibbsMotifFinder`.

**Initialization:** pick a random start in each sequence and store that k-mer as its initial motif. Valid starts run from 0 to `len(seq) - k`, so sequences of different lengths are handled.

**Iteration**, repeated up to 10,000 times:
1. Choose a sequence *i* at random (the "holdout").
2. Rebuild the PWM from the other N − 1 motifs with `build_pfm()` and `build_pwm()`. The pseudocount of 0.25 and the uniform 0.25 background come from `build_pwm()`.
3. Score every k-mer window in the full holdout sequence against the PWM with `score_kmer()`, on both the forward strand and the reverse complement.
4. Convert the log2 scores back to odds, normalize them into probabilities, $P(m) = \frac{A_m}{\sum_l A_l}$, and draw the new motif at random from that distribution. Reverse-strand picks are stored as reverse complements so every motif is in the same orientation.

**Convergence:** after each iteration, compute the information content of the full PFM with `pfm_ic()`. If the IC stays within 0.001 bits of the previous iteration for 50 iterations in a row, the motifs have stopped changing and we stop. Otherwise the loop ends at 10,000 iterations.

**Output:** the PFM built from the final set of motifs.


In [1]:
def GibbsMotifFinder(seqs, k, seed=None):
    """
    Function to find a pfm from a list of strings using a Gibbs sampler

    Args:
        seqs (str list): a list of sequences, not necessarily in same lengths
        k (int): the length of motif to find
        seed (int, default=None): seed for np.random

    Returns:
        pfm (numpy array): dimensions are 4xlength
    """
    # Use rng to make random samples/selections/numbers
    # Given code provided by the assignment
    random.seed(seed)
    rng = np.random.default_rng(seed)

    # Check for the length of k
    # Note: k cannot be longer than the sequences themselves
    # Use one line code to shorten code syntax
    too_short = [i for i, s in enumerate(seqs) if len(s) < k]  # i = index , s = sequence
    if too_short:
        raise ValueError(f"{len(too_short)} sequence(s) are shorter than k={k}")

    # Turns lowercase letters in the sequence to upper case
    seqs = [s.upper() for s in seqs]

    # INITIALIZATION
    # Random pick of k-length sequences from each line of DNA as motifs
    motifs = []
    for seq in seqs:
        # Randomize the start of each k-mer within each sequence
        start = rng.integers(0, len(seq) - k + 1)
        motifs.append(seq[start:start + k])

    # Convergence tracking
    convergence_streak = 0
    required_streak = 100         # Note from lecture: Determine if IC has changed within the first 100 iterations
    information_content = None    # IC from the previous iteration

    # ITERATION
    # Loop body: for j = 1 to 10000 or motifs stop changing (as seen as above)
    for j in range(10000):

        # Choose a sequence at random - the holdout
        i = rng.integers(0, len(seqs)) # i = holdout sequence

        # Rebuild the PWM from all motifs except motif i
        exclude_i = motifs[:i] + motifs[i + 1:]
        pfm = build_pfm(exclude_i, k)
        pwm = build_pwm(pfm)

        # Score each k-mer in the full holdout sequence, on both strands
        # m = start motif position of each window in the forward sequence
        forward_kmers = [seqs[i][m:m + k] for m in range(len(seqs[i]) - k + 1)]
        reverse_kmers = [reverse_complement(kmer) for kmer in forward_kmers]
        both_strands = forward_kmers + reverse_kmers
        scores = np.array([score_kmer(kmer, pwm) for kmer in both_strands])

        # Choose the new motif based on the score distribution
        # PWM scores are log2 odds, meaning it can have negatives and positives numbers
        # Normalize PWM scores to only be positive
        odds = 2 ** scores
        probability = odds / odds.sum()

        # Weighted random draw of position m with the reverse strands
        m = rng.choice(len(both_strands), p=probability)
        motifs[i] = both_strands[m]

        # CONVERGENCE
        # Stop once the IC has stayed within 0.001 bits for 50 iterations in a row
        ic = pfm_ic(build_pfm(motifs, k))

        if information_content is not None and np.isclose(ic, information_content, atol=1e-3):
            convergence_streak += 1
        else:
            convergence_streak = 0

        if convergence_streak >= required_streak:
            break

        information_content = ic

    # Build the final PFM from the final set of motifs
    return build_pfm(motifs, k)

In [ ]:
# Quick sanity check on a tiny example: each sequence contains GGGACGT,
# so the PFM should show that motif or part of its reverse complement (ACGTCCC)
test_seqs = ["GATTACAGGGACGTTTT", "CCCGGGACGTAAATTCC", "TTACGGGACGTGGCCAA"]
test_pfm = GibbsMotifFinder(test_seqs, 7, seed=100)
print(test_pfm)

In [ ]:
# Here we test your Gibbs sampler.
# You do not need to edit this or the section below. This is the Driver program

#read promoters, store in a list of strings
seq_file="data/GCF_000009045.1_ASM904v1_genomic.fna"
gff_file="data/GCF_000009045.1_ASM904v1_genomic.gff"

seqs = []

for name, seq in get_fasta(seq_file): # For each entry in our FASTA file
    for gff_entry in get_gff(gff_file): # For each entry in our GFF file
        if gff_entry.type == 'CDS': # If this is a coding sequence
            promoter_seq = get_seq(seq, gff_entry.start, gff_entry.end, gff_entry.strand, 50) # Extract 50 bp as a promoter
    
            """
            Because the gibbs sampling assumption is broken in just using promoters,
            and because it takes very long time to randomly progress through so many
            regions, for this example we will pre-filter for sequences that all contain
            part of the shine-dalgarno motif:
            """
            if "AGGAGG" in promoter_seq:
                seqs.append(promoter_seq)

---
# Driver Program
Don't change any of the code here. If you have completed the project by following the coding by contract, the following code should work.

In [ ]:
# Run the gibbs sampler:
promoter_pfm = GibbsMotifFinder(seqs,10 )

# Plot the final pfm that is generated: 
seqlogo.seqlogo(seqlogo.CompletePm(pfm = promoter_pfm.T))

---
# Challenge Yourself
Now that you potentially have a working skeleton of the project, try to work with the new NRF1 ChIP-seq data: `data/nrf1_gibbs.fa`.

Upon inspection, you will see that it is just a FASTA file. Since it is a FASTA file, you already have an example in the code above on how to access FASTA files. It is up to you to do the correct data ingest with it, though.

---
# NRF1 Driver Program
Modify the code below and test your Gibbs Sampler.

In [ ]:
# Here we test your Gibbs sampler.
# You do not need to edit this or the section below. This is the Driver program

#read promoters, store in a list of strings
nrf1_file="data/nrf1_gibbs.fa"

nrf1_peaks = []

# Data ingest of nrf1_peaks:
# Each FASTA entry is one 75 bp ChIP-seq peak; we only need the sequence
for name, seq in get_fasta(nrf1_file):
    seq = seq.upper() # Capitlizes the sequence
    if "GCATGC" in seq: # Chose a random motif
        nrf1_peaks.append(seq)

# Take a reproducible random subset so each peak is resampled many times
random.seed(2)
nrf1_peaks = random.sample(nrf1_peaks, 500)

# Run the gibbs sampler:
nrf1_pfm = GibbsMotifFinder(nrf1_peaks, 10)

# Plot the final pfm that is generated: 
seqlogo.seqlogo(seqlogo.CompletePm(pfm = nrf1_pfm.T))